In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')


import torch
import os
from transformer_lens import HookedTransformer
from transformers import AutoTokenizer
from dictionary_learning.mask_scae import SCAESuite, MergedSCAESuite # Your SCAESuite class
from datasets import load_dataset
from dictionary_learning.buffer import chunk_and_tokenize
from interp.interp_utils_new import (
    load_tokenized_dataset,
    collect_activations_and_tokens,
    generate_feature_dashboard
)

torch.set_grad_enabled(False)

# --- Configuration ---
MODEL_NAME = "EleutherAI/pythia-70m"
PATH_TO_PILE = "/root/dictionary_learning/pile-uncopyrighted"
# For local pile, you might need to specify data_files if it's a collection of .jsonl.gz
# e.g., data_files = {"train": ["/path/to/pile/train/00.jsonl.gz", ...]}
# For HF streaming, data_files can be None.
DATA_FILES = None # Set to list of local file paths if using local pile and not streaming
STREAM_PILE = True # If True, streams from HF. If False and DATA_FILES is None, loads non-streamed from HF.
                   # If DATA_FILES is set, STREAM_PILE=True will stream from those local files.
SEQ_LEN = 128
NUM_SAMPLES_TO_LOAD = 10000 # How many sequences to prepare from the dataset
NUM_SAMPLES_TO_PROCESS = 10000 # How many sequences to run through the suite for activations (max_batches_to_process * batch_size)
BATCH_SIZE_COLLECT = 64
OUTPUT_ACTIVATIONS_DIR = "output_activations_pile"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

/root/dictionary_learning/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- Initialize Model and Tokenizer ---
model = HookedTransformer.from_pretrained(MODEL_NAME, device=DEVICE)
model.eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# --- Load or Create your SCAESuite ---
# Option 1: Load from a checkpoint (replace with your actual loading logic)
# suite = SCAESuite.from_pretrained(SUITE_REPO_ID, model, device=DEVICE, dtype=torch.float32)

# Option 2: Initialize a new suite (for testing the pipeline if you don't have a trained one)
# This is a placeholder, ensure parameters match your actual suite setup
suite = SCAESuite.from_pretrained(
    repo_id="jacobcd52/pythia-70m_mask0_fact_fvu0_fvu_sparse0_fvu1.0_lr0.0005",
    model=model,
    device=DEVICE,
)

print(f"Using device: {DEVICE}")
os.makedirs(OUTPUT_ACTIVATIONS_DIR, exist_ok=True)

Loaded pretrained model EleutherAI/pythia-70m into HookedTransformer
Using device: cuda


In [30]:
raw_dataset = load_dataset(
    PATH_TO_PILE,
    split=f"train[:10%]",
)

raw_dataset = raw_dataset.shuffle(seed=42)


tokenized_dataset = chunk_and_tokenize(
    dataset=raw_dataset,
    tokenizer=tokenizer,
    text_key="text",
    max_length=SEQ_LEN,
    num_proc=max(1, os.cpu_count() // 2),
    load_from_cache_file=True
)

if len(tokenized_dataset) > NUM_SAMPLES_TO_PROCESS:
    dataset_to_process = tokenized_dataset.select(range(NUM_SAMPLES_TO_PROCESS))
else:
    dataset_to_process = tokenized_dataset


# Quick check of a sample
sample = next(iter(dataset_to_process))
if isinstance(sample["input_ids"], torch.Tensor):
    print("Sample token IDs shape:", sample["input_ids"].shape)
else: # It's a list for streaming datasets before DataLoader
    print("Sample token IDs length (list):", len(sample["input_ids"]))
    # If you need it as a tensor for this check:
    # sample_tensor = torch.tensor(sample["input_ids"])
    # print("Sample token IDs shape (converted to tensor):", sample_tensor.shape)

# Decoding should still work fine as tokenizer.decode can handle lists of IDs
print("Sample tokens (decoded):", tokenizer.decode(sample["input_ids"]))

Filter (num_proc=48): 100%|██████████| 589922/589922 [00:02<00:00, 222173.43 examples/s]


Sample token IDs shape: torch.Size([128])
Sample tokens (decoded): “We’re really rethinking how we understand urban space,” said Cagney, deputy dean of the Division of the Social Sciences and Professor of Sociology. “A concern among social scientists is that you have.

Lee, a social sciences master lecturer at Boston University. She later attended Boston University where she earned her.

The Bachelor of Arts in Sociology allows you to apply major sociological theories. Social Sciences: Select 9 credits from three different areas: Criminal Justice,

Call for Papers Vol. 9 No. 5 Submission Deadline: April 30, 2019. Aims and Scope.


In [83]:
toks_list = []
iterable_dataset = iter(dataset_to_process)
for _ in range(128):
    toks_list.append(next(iterable_dataset)['input_ids'])
toks = torch.stack(toks_list[:64], dim=0)
toks_test = torch.stack(toks_list[64:], dim=0)

layer = 3
ae = suite.module_dict[f'attn_{layer}'].ae
ae.k = 64
hook_pt = f'blocks.{layer}.hook_attn_out'
_, cache = model.run_with_cache(toks, return_type="loss", names_filter=[hook_pt])
act = cache[hook_pt]
recons = ae(act)
f = ae.encode(act)
fvu = (act - recons).pow(2).sum() / (act - act.mean(dim=[0])).pow(2).sum()

print("SAE FVU: ", fvu)

_, cache_test = model.run_with_cache(toks_test, return_type="loss", names_filter=[hook_pt])
act_test = cache_test[hook_pt]

# Compute PCA on training activations
q = 100
U, S, V = torch.pca_lowrank(act.reshape(-1, act.shape[-1]), q=q)
# Project test activations onto PCA components
act_test_flat = act_test.reshape(-1, act_test.shape[-1])
act_test_proj = act_test_flat @ V
# Compute variance explained
total_var = (act_test_flat - act_test_flat.mean(dim=[0])).pow(2).sum()
explained_var = (act_test_proj - act_test_proj.mean(dim=[0])).pow(2).sum()
print(f"FVU of top {q} PCA components: {1 - explained_var/total_var:.3f}")

SAE FVU:  tensor(0.0889, device='cuda:0')
FVU of top 100 PCA components: 0.162


In [149]:
# --- Generate Feature Dashboard ---
# Choose which activations to analyze (sparse or non-sparse)
mode_to_analyze = "sparse_false" # or "sparse_false"
activations_path = os.path.join(OUTPUT_ACTIVATIONS_DIR, mode_to_analyze)

# Specify the module and feature index you want to inspect
# Example: first attention module (attn_0), feature index 123
# You'll need to know the valid module names and feature ranges for your suite
module_to_inspect = "attn_3"


# Check if the activations path for the chosen mode exists
if not os.path.exists(activations_path):
    print(f"Activations path {activations_path} does not exist. Run collection for this mode first.")
else:
    # Check if there are any batch folders in the activations_path
    batch_folders_exist = any(d.startswith("batch_") for d in os.listdir(activations_path))
    if not batch_folders_exist:
        print(f"No batch data found in {activations_path}. Ensure activation collection was successful.")
    else:
        print(f"Generating dashboard using activations from: {activations_path}")
        generate_feature_dashboard(
            module_name_str=module_to_inspect,
            feature_idx_in_module=16,
            activations_base_dir=activations_path,
            tokenizer=tokenizer,
            model=model,
            suite=suite,
            k_top_contexts=20,
            context_window_size=256 # Number of tokens around the max activating one
        )

Generating dashboard using activations from: output_activations_pile/sparse_false
Generating dashboard for: attn_3, Feature Index: 16


In [142]:
s = """Welcome back!<br>Sign in to start taking action.<br><br>Thanks for signing up as a global citizen. In order to create your account we need you to provide your email address. You can check out our Privacy Policy to see how we safeguard and use the information you provide us with. If your Facebook account does not have an attached e-mail address, you'll need to add that before you can sign up."""

In [143]:
s = s.replace("<br>", "\n")

In [144]:
hook_pt = 'blocks.3.hook_attn_out'

In [145]:
_, cache = model.run_with_cache(s, return_type="loss", names_filter=[hook_pt])
act = cache[hook_pt]
f = suite.module_dict[f'attn_3'].ae.encode(act)

In [146]:
a = f[0, :, 14]
str_toks = model.to_tokens(s)[0, :]
for act, tok in zip(a, str_toks):
    x = "   <------------------" if act>0 else ""
    print(f"{act} {model.tokenizer.decode(tok)} {x}")


0.0 <|endoftext|> 
0.0 Welcome 
0.0  back 
0.0 ! 
0.0 
 
0.0 Sign 
0.0  in 
0.0  to 
0.0  start 
0.0  taking 
0.0  action 
0.0 . 
0.0 
 
0.0 
 
0.0 Thanks 
0.0  for 
0.0  signing 
0.0  up 
0.0  as 
0.0  a 
0.0  global 
0.0  citizen 
0.0 . 
0.0  In 
0.0  order 
0.0  to 
0.0  create 
0.0  your 
0.0  account 
0.0  we 
0.0  need 
0.0  you 
0.0  to 
0.0  provide 
0.0  your 
0.0  email 
0.0  address 
0.0 . 
0.0  You 
0.0  can 
0.0  check 
0.0  out 
0.0  our 
0.0  Privacy 
0.0  Policy 
0.0  to 
0.0  see 
0.0  how 
0.0  we 
0.0  safeguard 
0.0  and 
0.0  use 
0.0  the 
0.0  information 
0.0  you 
0.0  provide 
0.0  us 
0.0  with 
0.0 . 
0.0  If 
0.0  your 
0.0  Facebook 
0.0  account 
0.0  does 
0.0  not 
0.0  have 
0.0  an 
0.0  attached 
0.0  e 
0.0 - 
0.0 mail 
0.0  address 
0.0 , 
0.0  you 
0.0 'll 
0.0  need 
0.0  to 
0.0  add 
0.0  that 
0.0  before 
0.0  you 
0.0  can 
0.0  sign 
0.0  up 
0.0 . 


In [147]:
(a!=0).sum()

tensor(0, device='cuda:0')